# Theta-gamma coupling states in REM — PFC vs HPC

End-to-end re-implementation of the analyses in **Zhang et al. 2019, eLife,
"Sub-second dynamics of theta-gamma coupling in hippocampal CA1"**,
adapted to our PFC + HPC RGS data and stratified by **phasic vs tonic REM**
(no spike-based analyses are run because spikes are not available).

Methods reused verbatim from the paper:

1. **Wavelet spectrum normalized by theta phase** — Morlet CWT on HPC LFP
   at the native fs (1000 Hz), sequential time/frequency boxcar smoothing
   (±8 ms × ±2 Hz, matching Zhang's `ntw = 11`, `nsw = 3`), z-score per
   frequency, then 20 equal theta-phase bins → **FPP** = 81 × 20 matrix
   per cycle.

   *Project-specific change*: instead of Zhang's Butterworth + Hilbert
   theta-cycle detector, theta cycles are detected with the project's
   **mask-sift EMD pipeline** (`extract_imfs_by_pt_intervals` →
   theta IMF → `get_cycle_data` → `get_cycles_with_conditions` with
   `is_good == 1, duration in (fs/12, fs/5), max_amp > p25(IA)`). The
   per-sample phase reference used for binning is
   `np.angle(hilbert(theta_imf))`. Cycle counts match
   `RGS_pt_counter.py`.
2. **k-means clustering** with **Pearson-correlation distance** (`D = 1 − r`)
   and **k = 4**. Sorting clusters into **S / M / EF / LF** gamma by
   **gravity frequency** (power-weighted mean) and **gravity phase**
   (power-weighted circular mean) of the >95% peak field, exactly per
   Zhang's `PhaseFreSort.m`.
3. **Intra- vs inter-cluster correlation** under 5-fold CV.
4. **Cross-rat assignment accuracy** with reference m-FPPs from another rat.
5. **Markov state transitions** (4 × 4) and state occurrences.
6. **LFP-LFP PPC** — wavelet cross-spectrum (PFC × HPC*), 81 × 20 phase-binned
   per cycle, V-statistic unbiased PPC across cycles.

Project-specific adaptations:

* **CA3-CA1 and EC-CA1** in Zhang are replaced by **PFC-HPC**.
* **Pre / maze / post-track** in Zhang is replaced by **phasic vs tonic
  REM** (`extract_pt_intervals` in `src/utils.py`).
* All PPC heatmaps use the **hot** colormap by user request.
* Convention: `phase_lag = angle(PFC * conj(HPC))`. Positive lag → PFC leads
  HPC at that (frequency, theta phase).

The pipeline modules live in `exploration/zhang_tg_states/`.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make project modules importable
HERE = os.getcwd()
for rel in ('.', '..', '../..'):
    cand = os.path.abspath(os.path.join(HERE, rel))
    if cand not in sys.path:
        sys.path.insert(0, cand)
for rel in ('exploration', '../exploration'):
    cand = os.path.abspath(os.path.join(HERE, rel))
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

from zhang_tg_states import (
    ANALYSIS_FS, FREQUENCIES, N_PHASE_BINS,
    PHASE_CENTERS_DEG, PHASE_CENTERS_RAD,
    STATE_NAMES,
    cluster_fpps_into_tg_states,
    fpps_from_lfp_segment,
    intra_inter_correlation, kfold_intra_inter,
    cross_dataset_accuracy, pairwise_cross_dataset,
    phasic_vs_tonic_transitions,
    cross_spectrum_for_segment, state_conditioned_ppc,
    ppc_phase_pooled,
)
from zhang_tg_states import plotting as zplot
from zhang_tg_states.data_pipeline import (
    BASE_PATH, RAT_GROUPS, process_rat, process_rat_emd,
)
from zhang_tg_states.emd_cycles import load_default_emd_config

print('frequencies:', FREQUENCIES[0], '...', FREQUENCIES[-1],
      f'({len(FREQUENCIES)} bins)')
print('phase bins:', N_PHASE_BINS, '|', 'analysis fs:', ANALYSIS_FS, 'Hz')
print('state names:', STATE_NAMES)
print('RAT_GROUPS:', RAT_GROUPS)


## 1 — Per-rat data ingestion (EMD-driven theta cycles)

`process_rat_emd(rat_id, cfg)` walks every condition folder under
`BASE_PATH`, loads HPC + PFC LFPs and the hypnogram for each session,
runs the project's full EMD pipeline:

  1. `extract_pt_intervals` -> phasic and tonic REM IntervalSets.
  2. `extract_imfs_by_pt_intervals` -> mask-sift EMD per interval.
  3. `choose_theta_imf_index` -> pick theta IMF (prefers index 5; falls
     back to the IMF whose mean frequency is closest to the center of
     5–12 Hz).
  4. `get_cycle_data` on the theta IMF -> per-sample IP / IF / IA + an
     `emd.cycles.Cycles` object.
  5. `get_cycles_with_conditions` filter:
     `is_good == 1` &
     `duration_samples < fs/5` &
     `duration_samples > fs/12` &
     `max_amp > p25(IA)`.
  6. Cycle (start, end) sample indices come from
     `cycles.get_inds_of_cycle(i)`.
  7. Per-sample wrapped phase = `np.angle(hilbert(theta_imf))` (used to
     bin each cycle's Morlet wavelet power into 20 equal phase bins).

This yields cycle counts that match `exploration/RGS_pt_counter.py`
exactly. No resampling: everything runs at the native fs (1000 Hz).

Adjust `RUN_RATS` to limit which rats are processed during development.


In [ ]:
# Toggle: which rats to process
RUN_RATS = {
    'positive': RAT_GROUPS['positive'],   # [3, 4, 7, 8]
    'control':  RAT_GROUPS['control'],    # [1, 2, 6, 9]
}

emd_cfg = load_default_emd_config()

results = {'positive': {}, 'control': {}}
for group_name, rat_ids in RUN_RATS.items():
    print(f'\n=== group: {group_name} ===')
    for rid in rat_ids:
        agg = process_rat_emd(rid, cfg=emd_cfg, verbose=True)
        if agg is None:
            continue
        results[group_name][rid] = agg

n_total = sum(len(v) for v in results.values())
print(f'\nLoaded {n_total} rat aggregates')
for g, d in results.items():
    print(f'  {g}: rats = {sorted(d.keys())}')


### Quick summary table

How many cycles per rat per substate?


In [ ]:
rows = []
for group, rats in results.items():
    for rid, agg in rats.items():
        n_phasic = int((agg.cycle_substates == 'phasic').sum())
        n_tonic  = int((agg.cycle_substates == 'tonic').sum())
        rows.append(dict(group=group, rat=rid,
                         n_cycles_phasic=n_phasic,
                         n_cycles_tonic=n_tonic,
                         n_sessions=len(agg.sessions),
                         total_cycles=len(agg.cycle_substates)))
cycle_summary = pd.DataFrame(rows).sort_values(['group','rat']).reset_index(drop=True)
cycle_summary


## 2 — TG-state clustering

Per-rat clustering is the most faithful reflection of Zhang's approach
(channels are clustered individually, and the result is then summarised
across animals). We cluster each rat's full set of REM cycles
(phasic + tonic together) and label the four clusters as S, M, EF, LF.


In [ ]:
per_rat_cluster = {'positive': {}, 'control': {}}
for group, rats in results.items():
    for rid, agg in rats.items():
        if len(agg.fpps) < 200:
            print(f'rat {rid}: only {len(agg.fpps)} cycles, skipping cluster')
            continue
        cl = cluster_fpps_into_tg_states(
            agg.fpps.fpps,
            agg.frequencies,
            agg.phase_centers_rad,
            k=4, random_state=0, n_init=20,
        )
        per_rat_cluster[group][rid] = cl
        print(f'rat {rid}: gravity_freqs = '
              f'{np.round(cl.gravity_freqs,1)} Hz, '
              f'gravity_phases = {np.round(np.degrees(cl.gravity_phases),0)} deg')


### m-FPP panel per rat

Each row shows the four sorted m-FPPs for one rat (S, M, EF, LF). The
white contour traces the >95% peak field; the triangle marks the gravity
centre.


In [ ]:
FPP_CMAP = 'hot'   # m-FPP colormap (user request)

group_to_show = 'positive'
rats_to_show = list(per_rat_cluster[group_to_show].keys())
if not rats_to_show:
    print('no clusters in', group_to_show)
else:
    fig, axes = plt.subplots(len(rats_to_show), 4,
                             figsize=(13, 3.0*len(rats_to_show)),
                             constrained_layout=True)
    if len(rats_to_show) == 1:
        axes = axes[None, :]
    for ri, rid in enumerate(rats_to_show):
        cl = per_rat_cluster[group_to_show][rid]
        phase_deg = np.degrees(cl.phase_centers_rad)
        for s in range(4):
            zplot.plot_fpp(
                cl.m_fpps[s], cl.frequencies, phase_deg, ax=axes[ri, s],
                cmap=FPP_CMAP,
                title=f'rat {rid} - {STATE_NAMES[s]}\n'
                      f'{cl.gravity_freqs[s]:.1f} Hz, '
                      f'{np.degrees(cl.gravity_phases[s]):+.0f} deg',
                gravity_freq=cl.gravity_freqs[s],
                gravity_phase_deg=float(np.degrees(cl.gravity_phases[s]) % 360),
                mask=cl.masks[s],
            )
    plt.show()


### m-FPP per rat, phasic vs tonic separated

For each rat, the four sorted m-FPPs are recomputed twice: once using
only its **phasic** cycles, once using only its **tonic** cycles. Each
rat is one figure with 2 rows (phasic top, tonic bottom) × 4 columns
(S, M, EF, LF). Gravity centre + 95%-peak field mask are recomputed on
each substate-specific m-FPP. Triangles → substate-specific gravity
(white-fill = phasic, black-fill = tonic).

A within-rat colour scale is used so phasic and tonic are directly
comparable.


In [ ]:
from zhang_tg_states import (
    mean_fpp_per_cluster_substate, gravity_features,
)

group_to_show_pt = 'positive'
rats_to_show_pt = list(per_rat_cluster[group_to_show_pt].keys())

for rid in rats_to_show_pt:
    cl = per_rat_cluster[group_to_show_pt][rid]
    agg = results[group_to_show_pt][rid]
    m_by_sub = mean_fpp_per_cluster_substate(
        agg.fpps.fpps, cl.labels, agg.cycle_substates, n_clusters=4,
    )
    # Per-substate gravity / mask
    feats_phasic, feats_tonic = [], []
    for s in range(4):
        feats_phasic.append(gravity_features(
            m_by_sub['phasic'][s], cl.frequencies, cl.phase_centers_rad))
        feats_tonic.append(gravity_features(
            m_by_sub['tonic'][s], cl.frequencies, cl.phase_centers_rad))

    # Within-rat shared colour range across all 8 panels
    stack = np.concatenate([m_by_sub['phasic'].ravel(),
                            m_by_sub['tonic'].ravel()])
    vmin = float(np.nanpercentile(stack, 1))
    vmax = float(np.nanpercentile(stack, 99))

    fig, axes = plt.subplots(2, 4, figsize=(13, 6),
                             constrained_layout=True, sharex=True, sharey=True)
    phase_deg = np.degrees(cl.phase_centers_rad)
    n_ph = int((agg.cycle_substates == 'phasic').sum())
    n_to = int((agg.cycle_substates == 'tonic').sum())
    for s in range(4):
        # phasic
        zplot.plot_fpp(
            m_by_sub['phasic'][s], cl.frequencies, phase_deg,
            ax=axes[0, s], cmap=FPP_CMAP, vmin=vmin, vmax=vmax,
            title=f'rat {rid}  {STATE_NAMES[s]}  phasic\n'
                  f'{feats_phasic[s]["gravity_freq"]:.1f} Hz, '
                  f'{np.degrees(feats_phasic[s]["gravity_phase"]):+.0f} deg',
            gravity_freq=feats_phasic[s]['gravity_freq'],
            gravity_phase_deg=float(np.degrees(feats_phasic[s]['gravity_phase']) % 360),
            mask=feats_phasic[s]['mask'],
        )
        # tonic
        zplot.plot_fpp(
            m_by_sub['tonic'][s], cl.frequencies, phase_deg,
            ax=axes[1, s], cmap=FPP_CMAP, vmin=vmin, vmax=vmax,
            title=f'rat {rid}  {STATE_NAMES[s]}  tonic\n'
                  f'{feats_tonic[s]["gravity_freq"]:.1f} Hz, '
                  f'{np.degrees(feats_tonic[s]["gravity_phase"]):+.0f} deg',
            gravity_freq=feats_tonic[s]['gravity_freq'],
            gravity_phase_deg=float(np.degrees(feats_tonic[s]['gravity_phase']) % 360),
            mask=feats_tonic[s]['mask'],
        )
    fig.suptitle(f'rat {rid}  (n_phasic={n_ph}, n_tonic={n_to})', fontsize=11)
    plt.show()


### Group-average m-FPP (all REM cycles)

The four sorted m-FPPs are averaged across all rats in each group
(`positive` = RGS14, `control` = WT). One figure with 2 rows × 4
columns. Gravity centre + mask are recomputed on the group-averaged
m-FPP.


In [ ]:
# Build group-averaged m-FPPs (all cycles)
group_m_fpps = {}    # group -> (4, n_freq, n_phase)
group_n_rats = {}
for group in ['positive', 'control']:
    rats = list(per_rat_cluster[group].keys())
    group_n_rats[group] = len(rats)
    if not rats:
        group_m_fpps[group] = None
        continue
    stack = np.stack([per_rat_cluster[group][r].m_fpps for r in rats], axis=0)
    group_m_fpps[group] = np.nanmean(stack, axis=0)

# Shared colour range so positive and control are directly comparable
flat = np.concatenate([v.ravel() for v in group_m_fpps.values() if v is not None])
vmin = float(np.nanpercentile(flat, 1))
vmax = float(np.nanpercentile(flat, 99))

fig, axes = plt.subplots(2, 4, figsize=(13, 6),
                         constrained_layout=True, sharex=True, sharey=True)
freqs = FREQUENCIES
phase_deg = PHASE_CENTERS_DEG
for gi, group in enumerate(['positive', 'control']):
    gm = group_m_fpps[group]
    n_rats = group_n_rats[group]
    if gm is None:
        for s in range(4):
            axes[gi, s].set_axis_off()
        continue
    for s in range(4):
        feats = gravity_features(gm[s], freqs, PHASE_CENTERS_RAD)
        zplot.plot_fpp(
            gm[s], freqs, phase_deg, ax=axes[gi, s],
            cmap=FPP_CMAP, vmin=vmin, vmax=vmax,
            title=f'{group} (n_rats={n_rats})  {STATE_NAMES[s]}\n'
                  f'{feats["gravity_freq"]:.1f} Hz, '
                  f'{np.degrees(feats["gravity_phase"]):+.0f} deg',
            gravity_freq=feats['gravity_freq'],
            gravity_phase_deg=float(np.degrees(feats['gravity_phase']) % 360),
            mask=feats['mask'],
        )
plt.show()


### Group-average m-FPP, phasic vs tonic separated

For each group: m-FPPs are recomputed per state and per substate within
each rat, then averaged across rats. Result = 4 rows (positive-phasic,
positive-tonic, control-phasic, control-tonic) × 4 columns (S, M, EF,
LF). Shared colour scale across the whole figure.


In [ ]:
# Per-rat per-substate m-FPPs, then average across rats per group
group_m_by_sub = {}    # group -> {'phasic': (4,nf,np), 'tonic': (4,nf,np)}
for group in ['positive', 'control']:
    rats = list(per_rat_cluster[group].keys())
    if not rats:
        group_m_by_sub[group] = None
        continue
    rat_ph = []
    rat_to = []
    for rid in rats:
        cl = per_rat_cluster[group][rid]
        agg = results[group][rid]
        m_by = mean_fpp_per_cluster_substate(
            agg.fpps.fpps, cl.labels, agg.cycle_substates, n_clusters=4,
        )
        rat_ph.append(m_by['phasic'])
        rat_to.append(m_by['tonic'])
    group_m_by_sub[group] = {
        'phasic': np.nanmean(np.stack(rat_ph, axis=0), axis=0),
        'tonic':  np.nanmean(np.stack(rat_to, axis=0), axis=0),
        'n_rats': len(rats),
    }

# Shared colour scale across the whole figure
all_arrays = []
for v in group_m_by_sub.values():
    if v is None:
        continue
    all_arrays.append(v['phasic'].ravel())
    all_arrays.append(v['tonic'].ravel())
flat = np.concatenate(all_arrays) if all_arrays else np.array([0.0, 1.0])
vmin = float(np.nanpercentile(flat, 1))
vmax = float(np.nanpercentile(flat, 99))

fig, axes = plt.subplots(4, 4, figsize=(13, 12),
                         constrained_layout=True, sharex=True, sharey=True)
row_layout = [
    ('positive', 'phasic'),
    ('positive', 'tonic'),
    ('control',  'phasic'),
    ('control',  'tonic'),
]
for ri, (group, substate) in enumerate(row_layout):
    block = group_m_by_sub.get(group)
    if block is None:
        for s in range(4):
            axes[ri, s].set_axis_off()
        continue
    arr = block[substate]
    n_rats = block['n_rats']
    for s in range(4):
        feats = gravity_features(arr[s], FREQUENCIES, PHASE_CENTERS_RAD)
        zplot.plot_fpp(
            arr[s], FREQUENCIES, PHASE_CENTERS_DEG, ax=axes[ri, s],
            cmap=FPP_CMAP, vmin=vmin, vmax=vmax,
            title=f'{group} {substate} (n={n_rats})\n'
                  f'{STATE_NAMES[s]}  {feats["gravity_freq"]:.1f} Hz, '
                  f'{np.degrees(feats["gravity_phase"]):+.0f} deg',
            gravity_freq=feats['gravity_freq'],
            gravity_phase_deg=float(np.degrees(feats['gravity_phase']) % 360),
            mask=feats['mask'],
        )
plt.show()


### Polar density of gravity centres

Aggregating across rats: each point is one rat's gravity centre for one
TG state. Angle = gravity phase, radius = gravity frequency.


In [ ]:
fig = plt.figure(figsize=(8, 4))
for gi, group in enumerate(['positive', 'control']):
    ax = fig.add_subplot(1, 2, gi+1, projection='polar')
    for s in range(4):
        freqs = [cl.gravity_freqs[s] for cl in per_rat_cluster[group].values()]
        phases = [cl.gravity_phases[s] for cl in per_rat_cluster[group].values()]
        if not freqs:
            continue
        ax.scatter(phases, freqs,
                   color=zplot.STATE_COLORS[s], label=STATE_NAMES[s],
                   s=80, alpha=0.7, edgecolor='black')
    ax.set_rlim(0, 180); ax.set_rticks([60,120,180])
    ax.set_title(f'{group}')
    if gi == 1:
        ax.legend(loc='upper right', bbox_to_anchor=(1.5, 1.05), fontsize=8)
plt.suptitle('Gravity centres per rat (angle = phase, radius = freq)')
plt.show()


## 3 — Intra- vs inter-cluster correlation (5-fold CV)

For each cycle, intra = correlation with its own state's mean FPP;
inter_max = best correlation with any *other* state's mean FPP. Most
cycles should sit well above the inter line; cycles whose gap is near
zero share features with multiple states (Zhang reports ~20%).


In [ ]:
ii_per_rat = {'positive': {}, 'control': {}}
for group, rats in per_rat_cluster.items():
    for rid, cl in rats.items():
        agg = results[group][rid]
        ii_per_rat[group][rid] = kfold_intra_inter(
            agg.fpps.fpps, agg.frequencies, agg.phase_centers_rad,
            n_folds=5, k=4, random_state=0,
        )

# Plot intra and inter distributions across all rats
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
for gi, group in enumerate(['positive', 'control']):
    intras = np.concatenate([r.intra for r in ii_per_rat[group].values()])
    inters = np.concatenate([r.inter_max for r in ii_per_rat[group].values()])
    axes[gi].hist(intras, bins=80, color='tab:blue', alpha=0.7,
                  label='intra-cluster')
    axes[gi].hist(inters, bins=80, color='tab:red', alpha=0.5,
                  label='max inter-cluster')
    axes[gi].set_xlabel('correlation r')
    axes[gi].set_ylabel('# cycles')
    axes[gi].set_title(f'{group}  (n={len(intras)})')
    axes[gi].legend()
plt.show()

# Gap distribution (intra - inter_max)
fig, ax = plt.subplots(figsize=(5, 3.2))
for group, color in [('positive','C0'), ('control','C1')]:
    gaps = np.concatenate([r.gap for r in ii_per_rat[group].values()])
    ax.hist(gaps, bins=80, alpha=0.55, color=color, label=group, density=True)
ax.axvline(0.0, color='k', lw=0.6, linestyle='--')
ax.axvline(0.05, color='gray', lw=0.6, linestyle=':')
ax.set_xlabel('intra - inter_max');  ax.set_ylabel('density')
ax.set_title('Cycle uniqueness to its assigned state')
ax.legend()
plt.show()


## 4 — Cross-rat assignment accuracy

Take rat $i$'s cycles, classify them with rat $j$'s reference m-FPPs.
Compare against rat $i$'s native cluster labels. Diagonal is 1.0 by
construction. Off-diagonal entries quantify how well a rat's TG state
definitions transfer.


In [ ]:
for group in ['positive', 'control']:
    rats = list(per_rat_cluster[group].keys())
    if len(rats) < 2:
        continue
    fpps_list = [results[group][r].fpps.fpps for r in rats]
    labels_list = [per_rat_cluster[group][r].labels for r in rats]
    m_fpps_list = [per_rat_cluster[group][r].m_fpps for r in rats]
    acc = pairwise_cross_dataset(fpps_list, labels_list, m_fpps_list)
    fig, ax = plt.subplots(figsize=(4.0, 3.4))
    im = ax.imshow(acc, vmin=0.25, vmax=1.0, cmap='viridis')
    ax.set_xticks(range(len(rats))); ax.set_yticks(range(len(rats)))
    ax.set_xticklabels([f'r{r}' for r in rats], rotation=0)
    ax.set_yticklabels([f'r{r}' for r in rats])
    ax.set_title(f'cross-rat accuracy ({group})')
    for i in range(len(rats)):
        for j in range(len(rats)):
            ax.text(j, i, f'{acc[i,j]:.2f}', ha='center', va='center',
                    fontsize=7,
                    color='white' if acc[i,j] < 0.6 else 'black')
    plt.colorbar(im, ax=ax, shrink=0.7)
    plt.show()
    off = acc[~np.eye(len(rats), dtype=bool)]
    print(f'{group}: off-diagonal accuracy = {off.mean():.2f} '
          f'(min {off.min():.2f}, max {off.max():.2f}, chance = 0.25)')


## 5 — Markov state transitions: phasic vs tonic

For each rat we tag every cycle as phasic, tonic, or excluded, build the
state sequence within each contiguous phasic / tonic interval, and
compute a 4×4 transition matrix per substate. Transitions across
substate boundaries are not counted.


In [ ]:
from zhang_tg_states import transitions_by_substate

trans_per_rat = {'positive': {}, 'control': {}}

for group, rats in per_rat_cluster.items():
    for rid, cl in rats.items():
        agg = results[group][rid]
        # Each agg.cycle_segment_ids[i] is a unique integer for the
        # phasic/tonic interval the cycle belongs to. Runs = groups of
        # cycles sharing a segment_id (= cycles within ONE REM substate
        # interval). This avoids reconstructing interval bounds from the
        # non-chronological concatenated cycle array.
        tt = transitions_by_substate(
            labels=cl.labels,
            substates=agg.cycle_substates,
            segment_ids=agg.cycle_segment_ids,
        )
        trans_per_rat[group][rid] = tt

# Plot per-group averaged transition matrices and occurrence bars
for group in ['positive', 'control']:
    rats = list(trans_per_rat[group].keys())
    if not rats:
        continue
    Tph = np.mean([trans_per_rat[group][r]['phasic'].transition_matrix for r in rats], axis=0)
    Tto = np.mean([trans_per_rat[group][r]['tonic'].transition_matrix for r in rats], axis=0)
    Oph = np.mean([trans_per_rat[group][r]['phasic'].occurrence for r in rats], axis=0)
    Oto = np.mean([trans_per_rat[group][r]['tonic'].occurrence for r in rats], axis=0)

    fig, axes = plt.subplots(1, 4, figsize=(15, 3.2), constrained_layout=True)
    zplot.plot_transition_matrix(Tph, ax=axes[0], title=f'{group} phasic (n_rats={len(rats)})')
    zplot.plot_transition_matrix(Tto, ax=axes[1], title=f'{group} tonic (n_rats={len(rats)})')
    zplot.plot_transition_diff(Tph, Tto, ax=axes[2], title='phasic - tonic')
    zplot.plot_state_occurrence({'phasic': Oph, 'tonic': Oto},
                                ax=axes[3], title=f'occurrence ({group})')
    plt.show()


## 6 — PFC-HPC PPC per TG state, phasic vs tonic

For each rat:

* split cycles by `substate` ∈ {phasic, tonic};
* within each subset, split by TG state {S, M, EF, LF};
* compute PPC across the cycles for each (substate × state) cell, then
  average PPC heatmaps across rats for the group-level plot.

Convention: `phase_lag = angle(PFC * conj(HPC))`. Higher PPC = more
reliable PFC→HPC phase lag at that (frequency, theta phase).


In [ ]:
def state_ppc_split(agg, labels):
    out = {}
    for substate in ['phasic', 'tonic']:
        mask = agg.cycle_substates == substate
        if not np.any(mask):
            out[substate] = None
            continue
        res = state_conditioned_ppc(agg.cross_spectrum.angles[mask],
                                    labels[mask],
                                    frequencies=agg.frequencies,
                                    phase_centers_deg=PHASE_CENTERS_DEG)
        out[substate] = res
    return out

per_rat_ppc = {'positive': {}, 'control': {}}
for group, rats in per_rat_cluster.items():
    for rid, cl in rats.items():
        agg = results[group][rid]
        per_rat_ppc[group][rid] = state_ppc_split(agg, cl.labels)

# Group means
def stack_group(group):
    arr_phasic = []
    arr_tonic = []
    nph = []
    nto = []
    for rid in per_rat_ppc[group]:
        ph = per_rat_ppc[group][rid]['phasic']
        to = per_rat_ppc[group][rid]['tonic']
        if ph is None or to is None:
            continue
        arr_phasic.append(ph.ppc_per_state)
        arr_tonic.append(to.ppc_per_state)
        nph.append(ph.n_cycles_per_state)
        nto.append(to.n_cycles_per_state)
    if not arr_phasic:
        return None, None, None, None
    return (np.nanmean(np.stack(arr_phasic, 0), axis=0),
            np.nanmean(np.stack(arr_tonic, 0), axis=0),
            np.sum(np.stack(nph, 0), axis=0),
            np.sum(np.stack(nto, 0), axis=0))

for group in ['positive', 'control']:
    g_ph, g_to, n_ph, n_to = stack_group(group)
    if g_ph is None:
        continue
    print(f'group {group}: cycles per state — phasic {n_ph}, tonic {n_to}')
    vmax = float(np.nanpercentile(np.concatenate([g_ph.ravel(), g_to.ravel()]), 99))
    fig = zplot.plot_phasic_vs_tonic_ppc(
        g_ph, g_to, FREQUENCIES, PHASE_CENTERS_DEG,
        vmin=0.0, vmax=vmax, title_prefix=f'PFC-HPC PPC — {group}',
    )
    plt.show()


### PPC(f) curves averaged in the gravity phase window

For each TG state, the PPC heatmap is summarised by averaging across the
gravity-centred phase window `[phase − 7σ, phase + σ]` (Zhang). The
result is a single PPC vs frequency curve per state.


In [ ]:
for group in ['positive', 'control']:
    rats = list(per_rat_cluster[group].keys())
    if not rats:
        continue

    # Compute PPC(f) per rat using THAT rat's own gravity phase and std
    # so the inter-rat std reflects honest variability (cluster boundaries
    # and PPC heatmap both jitter across animals).
    n_states = 4
    per_rat_ppc_f_phasic = []   # each entry: (n_states, n_freq)
    per_rat_ppc_f_tonic = []
    rat_labels = []
    for rid in rats:
        cl = per_rat_cluster[group][rid]
        pr = per_rat_ppc[group][rid]
        if pr['phasic'] is None or pr['tonic'] is None:
            continue
        ph = pr['phasic'].ppc_per_state    # (4, n_freq, n_phase)
        to = pr['tonic'].ppc_per_state
        gp = cl.gravity_phases             # (4,)
        psd = cl.phase_stds                # (4,)
        ppc_f_ph = np.zeros((n_states, len(FREQUENCIES)))
        ppc_f_to = np.zeros((n_states, len(FREQUENCIES)))
        for s in range(n_states):
            ppc_f_ph[s], _ = ppc_phase_pooled(ph[s], gp[s], psd[s],
                                              phase_centers_rad=PHASE_CENTERS_RAD)
            ppc_f_to[s], _ = ppc_phase_pooled(to[s], gp[s], psd[s],
                                              phase_centers_rad=PHASE_CENTERS_RAD)
        per_rat_ppc_f_phasic.append(ppc_f_ph)
        per_rat_ppc_f_tonic.append(ppc_f_to)
        rat_labels.append(rid)
    if not per_rat_ppc_f_phasic:
        continue

    stack_ph = np.stack(per_rat_ppc_f_phasic, axis=0)  # (n_rats, n_states, n_freq)
    stack_to = np.stack(per_rat_ppc_f_tonic, axis=0)
    mean_ph = np.nanmean(stack_ph, axis=0)
    std_ph  = np.nanstd(stack_ph, axis=0, ddof=1) if len(rat_labels) > 1 else np.zeros_like(mean_ph)
    mean_to = np.nanmean(stack_to, axis=0)
    std_to  = np.nanstd(stack_to, axis=0, ddof=1) if len(rat_labels) > 1 else np.zeros_like(mean_to)

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), sharey=True,
                             constrained_layout=True)
    zplot.plot_ppc_per_freq(mean_ph, FREQUENCIES,
                            ax=axes[0],
                            title=f'{group} - phasic  (n_rats = {len(rat_labels)})',
                            ppc_per_state_freq_std=std_ph)
    zplot.plot_ppc_per_freq(mean_to, FREQUENCIES,
                            ax=axes[1],
                            title=f'{group} - tonic  (n_rats = {len(rat_labels)})',
                            ppc_per_state_freq_std=std_to)
    axes[0].set_ylabel('PPC')
    plt.show()


## 7 — Persist the artifacts

Save the per-rat cluster results, validation results, transition matrices,
and PPC matrices to disk for downstream analysis.


In [ ]:
out_dir = os.path.join(HERE, 'zhang_tg_states')
os.makedirs(out_dir, exist_ok=True)

def to_dict(cl):
    return dict(
        labels=cl.labels, m_fpps=cl.m_fpps,
        gravity_freqs=cl.gravity_freqs, gravity_phases=cl.gravity_phases,
        freq_stds=cl.freq_stds, phase_stds=cl.phase_stds,
        masks=cl.masks, raw_labels=cl.raw_labels, permutation=cl.permutation,
        frequencies=cl.frequencies, phase_centers_rad=cl.phase_centers_rad,
    )

def ppc_to_dict(d):
    return {k: (None if v is None else dict(
        ppc_per_state=v.ppc_per_state,
        n_cycles_per_state=v.n_cycles_per_state,
        n_per_freq_phase=v.n_per_freq_phase,
        frequencies=v.frequencies,
        phase_centers_deg=v.phase_centers_deg,
    )) for k, v in d.items()}

def trans_to_dict(d):
    return {k: dict(counts=v.counts,
                    transition_matrix=v.transition_matrix,
                    occurrence=v.occurrence,
                    n_cycles=v.n_cycles,
                    n_transitions=v.n_transitions) for k, v in d.items()}

for group in ['positive', 'control']:
    payload = dict(
        group=group,
        rat_ids=list(per_rat_cluster[group].keys()),
        clusters={r: to_dict(c) for r, c in per_rat_cluster[group].items()},
        intra_inter={r: dict(intra=v.intra, inter_max=v.inter_max,
                              labels=v.labels, gap=v.gap)
                     for r, v in ii_per_rat[group].items()},
        transitions={r: trans_to_dict(v)
                     for r, v in trans_per_rat[group].items()},
        ppc={r: ppc_to_dict(v) for r, v in per_rat_ppc[group].items()},
        cycle_summary=cycle_summary.to_dict(orient='records'),
        meta=dict(
            frequencies=FREQUENCIES, phase_centers_deg=PHASE_CENTERS_DEG,
            phase_centers_rad=PHASE_CENTERS_RAD, analysis_fs=ANALYSIS_FS,
            state_names=STATE_NAMES,
            source='PFC', target='HPC',
        ),
    )
    fname = os.path.join(out_dir, f'tg_states_{group}_pfc_hpc.pkl')
    with open(fname, 'wb') as fh:
        pickle.dump(payload, fh)
    print('wrote', fname)


## 8 — Distribution of PPC values: phasic vs tonic

How spread out are the PPC values themselves across `(frequency, theta-phase)`
bins? If phasic REM produces *irregular* coupling — only a few hot spots
in the heatmap — then the phasic PPC distribution should have a wide
spread (some near-zero, some very high). If tonic REM produces a more
*regular*, uniform coupling, the tonic distribution should be tighter
around its mean.

For each group and each TG state we pool every PPC value across
`(rats × frequencies × phase-bins)` and plot the **density** for phasic
vs tonic side by side. Dashed verticals mark each substate's mean. The
title quotes mean (μ), std (σ), and IQR — a tighter σ means a more
regular coupling pattern; a long right tail means a few very
strongly-locked bins dominate the heatmap.

The right-most panel ('all states') pools across the four TG states so
you can see the global substate effect on coupling regularity.


In [ ]:
for group in ['positive', 'control']:
    rats = list(per_rat_ppc[group].keys())
    if not rats:
        continue

    # pooled[substate][state] = list of (n_freq * n_phase,) arrays, one per rat
    pooled = {sub: {s: [] for s in range(4)} for sub in ('phasic', 'tonic')}
    for rid in rats:
        pr = per_rat_ppc[group][rid]
        for substate in ('phasic', 'tonic'):
            if pr[substate] is None:
                continue
            for s in range(4):
                pooled[substate][s].append(pr[substate].ppc_per_state[s].ravel())

    flat = {sub: {s: (np.concatenate(pooled[sub][s])
                       if pooled[sub][s] else np.array([]))
                  for s in range(4)}
            for sub in ('phasic', 'tonic')}
    for sub in ('phasic', 'tonic'):
        flat[sub]['all'] = (np.concatenate([flat[sub][s] for s in range(4)])
                            if any(flat[sub][s].size for s in range(4))
                            else np.array([]))

    cols = list(range(4)) + ['all']
    titles = list(STATE_NAMES) + ['all states pooled']

    fig, axes = plt.subplots(1, 5, figsize=(17, 3.6),
                             sharey=False, constrained_layout=True)
    for ci, (col, title) in enumerate(zip(cols, titles)):
        ph = flat['phasic'][col]; ph = ph[np.isfinite(ph)]
        to = flat['tonic'][col];  to = to[np.isfinite(to)]
        if ph.size == 0 or to.size == 0:
            axes[ci].set_axis_off()
            continue
        all_vals = np.concatenate([ph, to])
        vmin, vmax = np.nanpercentile(all_vals, [0.5, 99.5])
        bins = np.linspace(vmin, vmax, 60)
        axes[ci].hist(ph, bins=bins, color='tab:blue', alpha=0.55,
                      density=True, label='phasic')
        axes[ci].hist(to, bins=bins, color='tab:orange', alpha=0.55,
                      density=True, label='tonic')
        axes[ci].axvline(np.mean(ph), color='tab:blue',
                          linestyle='--', lw=1.0)
        axes[ci].axvline(np.mean(to), color='tab:orange',
                          linestyle='--', lw=1.0)
        iqr_ph = np.subtract(*np.percentile(ph, [75, 25]))
        iqr_to = np.subtract(*np.percentile(to, [75, 25]))
        axes[ci].set_title(
            f'{title}\n'
            f'phasic: μ={np.mean(ph):.3f}, σ={np.std(ph):.3f}, IQR={iqr_ph:.3f}\n'
            f'tonic:  μ={np.mean(to):.3f}, σ={np.std(to):.3f}, IQR={iqr_to:.3f}',
            fontsize=8,
        )
        axes[ci].set_xlabel('PPC')
        if ci == 0:
            axes[ci].set_ylabel('density')
        if ci == len(cols) - 1:
            axes[ci].legend(fontsize=8, frameon=False)
    fig.suptitle(f'PPC value distribution — {group}'
                 f'  (n_rats = {len(rats)})', fontsize=12)
    plt.show()


### Interpreting the σ and IQR

* **σ (std) and IQR small** → PPC values across the heatmap cluster
  tightly around the mean. Coupling is *regular*: most (f, θ) bins have
  similar PPC.
* **σ and IQR large**, or a heavy right tail in the histogram →
  coupling is *irregular*: a few bins are strongly locked while most
  are near zero.

Differences in mean answer "is coupling stronger overall?"; differences
in σ/IQR answer "is coupling more concentrated in specific (f, θ) bins,
or more uniform?". The two are independent — phasic could have the
same mean as tonic but a fatter tail, or vice versa.


## Interpretation summary

* **Wavelet FPPs** capture the joint distribution of gamma power and
  theta phase per individual cycle, exactly as in Zhang Fig. 1B.
* The four TG states sort by gravity frequency (S < M < EF ≈ LF) and
  phase (S, M near θ peak; EF early descending; LF late descending).
* **Intra/inter correlation under 5-fold CV** quantifies how "pure"
  each cycle is. A near-zero gap means a cycle has features of more
  than one TG state.
* **Cross-rat accuracy** shows whether m-FPPs transfer between rats —
  consistently >> 0.25 (chance) means the four states are robust across
  animals.
* **Markov transitions** reveal whether the network *switches* states
  rapidly or persists, separately for phasic vs tonic REM. The
  occurrence bars and the (phasic − tonic) difference matrix highlight
  REM-substate-specific reorganisations.
* **PFC-HPC PPC** per TG state shows whether the inter-area phase lag
  is reliable in each oscillatory regime. Differences between phasic
  and tonic suggest substate-specific PFC↔HPC coordination — the
  question this notebook is built to answer.

This pipeline is a faithful adaptation of Zhang et al. 2019 with two
substantive substitutions: PFC for CA3/EC, and phasic/tonic REM for
pre/track/post.
